# ThermalGAN single-sequence variant generation (Colab)

<img src="https://raw.githubusercontent.com/Umbriona/THOR/main/Covver_image_small.png" height="200" align="right" style="height:240px">


This notebook takes a input protein sequence of length 50-512,
And runs ESM + ThermalGAN, and writes a FASTA of optimized variants.


## 0) Runtime
Set Runtime -> Change runtime type -> GPU before running.


In [1]:
#@title Clone repository (optional)
from pathlib import Path

GIT_REPO_URL = "https://github.com/Umbriona/THOR.git"  # TODO: set to your repo URL
REPO_DIR = Path("/content/thermalgan-repo")

if GIT_REPO_URL:
    if not REPO_DIR.exists():
        !git clone -q {GIT_REPO_URL} {REPO_DIR}
    else:
        print(f"Repo already exists at {REPO_DIR}")
else:
    print("Skipping clone. Set GIT_REPO_URL if you want to clone.")


In [2]:
#@title Install dependencies
%pip -q install  "transformers>=4.36" "biopython>=1.79" "pyyaml" "tqdm" "typeguard" "gdown"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.6 MB/s eta 0:00:00


In [3]:
#@title Configure paths
import sys
from pathlib import Path

candidates = []
if "REPO_DIR" in globals() and REPO_DIR.exists():
    candidates.extend([REPO_DIR / "sandra" / "ThermalGAN", REPO_DIR ])
candidates.extend([Path("/content/ThermalGAN"), Path("/content/sandra/ThermalGAN")])

THERMALGAN_ROOT = None
for c in candidates:
    if c.exists():
        THERMALGAN_ROOT = c
        break

if THERMALGAN_ROOT is None:
    raise FileNotFoundError("Could not locate ThermalGAN. Set THERMALGAN_ROOT manually.")

sys.path.append(str(THERMALGAN_ROOT / "src"))
sys.path.append(str(THERMALGAN_ROOT / "src/scripts"))

print("THERMALGAN_ROOT:", THERMALGAN_ROOT)


THERMALGAN_ROOT: /content/thermalgan-repo


## 1) Weights
You need a run directory that contains `config.yaml` and `weights/epoch_<n>/generator_G.h5`.
Optionally, download OGT weights if you want predicted OGT values.


In [4]:
#@title Download weights (optional) from Hugging Face
from pathlib import Path
from huggingface_hub import snapshot_download

def extract(archive, dest_dir):
    if str(archive).endswith(".zip"):
        !unzip -q {archive} -d {dest_dir}
    elif str(archive).endswith((".tar.gz", ".tgz")):
        !tar -xzf {archive} -C {dest_dir}
    else:
        print("Unknown archive extension; extract manually if needed.")


DOWNLOAD_DIR = Path("/content/thermalgan_weights")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

REPO_ID = "Umbriona/THOR"   # your HF repo

# Option 1: download EVERYTHING in the repo
snapshot_download(
    repo_id=REPO_ID,
    repo_type="model",                 # HF "model" repo (default)
    local_dir=str(DOWNLOAD_DIR),
    local_dir_use_symlinks=False,      # better for Colab
    resume_download=True,
)
# Extract OGT weights
arch = DOWNLOAD_DIR / "OGT.tar.gz"
extract(arch, DOWNLOAD_DIR )
print("Downloaded to:", DOWNLOAD_DIR)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Downloaded to: /content/thermalgan_weights


In [5]:
#@title Configure run_dir and output
from pathlib import Path

RUN_DIR = Path("/content/thermalgan_weights/")
WEIGHTS = ""

OUTPUT_DIR = Path("/content/thermalgan_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OGT_WEIGHTS_DIR = Path("/content/thermalgan_weights")
OGT_CONFIG = RUN_DIR / "config_classifier1.yaml"

if not (RUN_DIR / "config.yaml").exists():
    raise FileNotFoundError(f"Missing config.yaml in {RUN_DIR}")
if not (RUN_DIR / "generator_G.h5").exists():
    raise FileNotFoundError("Missing generator_G.h5 for the selected epoch")


In [6]:
#@title Sequence input and generation settings
sequence = "MRKKVTIVGAGNVGATAAHWIASKELADVVLIDVVEGVPQGKGLDLLEAMPIEKRDAHILGTNDYADTANSDIVVITAGIARKPGMSRDDLLQTNYKIMSDVVGKAIAASPDCILIIVSNPLDAMAQTAYRQAKFNRNRVIGMAGVLDSARFRTFIAEELKVSVENVTAFVLGGHGDTMVPLARYSTVAGIPITELIPKERLDAIVQRTRDGGAEIVKYLKTGSAYYAPSAAAVEMVEAILKDKKKILPCAAFLEGEYGISGYYIGVPCKLGARGLEQIIEIRLTPEEEAALKKSAEAVKELCAVIGV" #@param {type:"string"}
#@markdown
sequence_id = "query1" #@param {type:"string"}
sequence_temp = None  # optional numeric temperature
output_name = sequence_id

replicates = 5 #@param {type:"integer"}
temperature = 1.0 #@param {type:"slider", min:0.1, max:1, step:0.01}


esm_init_mode = "single" #@param ["random", "single"]
esm_random_mask_prob = 0.15
esm_random_chunk = 8
esm_random_seed = 42

esm_filter_threshold = 3.0 #@param {type:"slider", min:1, max:4, step:0.05}
filter_opt_cycles = 0 #@param {type:"integer"}
sampling_cycles = 0 #@param {type:"integer"}
skip_optimize = False #@param {type:"boolean"}

hf_model = "facebook/esm1v_t33_650M_UR90S_1"
esm_device = "cuda"  # cuda or cpu
batch_tokens = 20000
max_length = 1022
esm_bf16 = True
gpu_id = 0

skip_ogt = False #@param {type:"boolean"}
store_softmax = False #@param {type:"boolean"}


In [7]:
#@title Run ThermalGAN
import shlex
import subprocess
from subprocess import CalledProcessError

sequence = "".join(sequence.split())
if not sequence:
    raise ValueError("Sequence is empty.")

cmd = [
    "python",
    str(THERMALGAN_ROOT / "src/scripts/generate_thor.py"),
    "--run_dir", str(RUN_DIR),
    "--sequence", sequence,
    "--seq_id", sequence_id,
    "--output_dir", str(OUTPUT_DIR),
    "--name", output_name,
    "--replicates", str(replicates),
    "--temperature", str(temperature),
    "--hf_model", hf_model,
    "--device", esm_device,
    "--batch_tokens", str(batch_tokens),
    "--max_length", str(max_length),
    "--gpu", str(gpu_id),
    "--esm_init_mode", esm_init_mode,
    "--esm_random_mask_prob", str(esm_random_mask_prob),
    "--esm_random_chunk", str(esm_random_chunk),
    "--esm_random_seed", str(esm_random_seed),
    "--filter_opt_cycles", str(filter_opt_cycles),
    "--sampling_cycles", str(sampling_cycles),
]
if sequence_temp is not None:
    cmd += ["--seq_temp", str(sequence_temp)]
if store_softmax:
    cmd += ["--store_softmax"]
if esm_filter_threshold is not None:
    cmd += ["--esm_filter_threshold", str(esm_filter_threshold)]
if skip_optimize:
    cmd += ["--skip_optimize"]
if esm_bf16:
    cmd += ["--esm_bf16"]
if skip_ogt:
    cmd += ["--skip_ogt"]
else:
    cmd += ["--ogt_config", str(OGT_CONFIG)]
    cmd += [
        "--ogt_weights",
        str(DOWNLOAD_DIR / "OGT" / "Model1" / "model_weights.h5"),
        str(DOWNLOAD_DIR / "OGT" / "Model2" / "model_weights.h5"),
        str(DOWNLOAD_DIR / "OGT" / "Model3" / "model_weights.h5"),
    ]

print("Running:", " ".join(shlex.quote(c) for c in cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print("Return Code:", result.returncode)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)

if result.returncode != 0:
    raise CalledProcessError(result.returncode, cmd, output=result.stdout, stderr=result.stderr)


Running: python /content/thermalgan-repo/src/scripts/generate_thor.py --run_dir /content/thermalgan_weights --sequence MRKKVTIVGAGNVGATAAHWIASKELADVVLIDVVEGVPQGKGLDLLEAMPIEKRDAHILGTNDYADTANSDIVVITAGIARKPGMSRDDLLQTNYKIMSDVVGKAIAASPDCILIIVSNPLDAMAQTAYRQAKFNRNRVIGMAGVLDSARFRTFIAEELKVSVENVTAFVLGGHGDTMVPLARYSTVAGIPITELIPKERLDAIVQRTRDGGAEIVKYLKTGSAYYAPSAAAVEMVEAILKDKKKILPCAAFLEGEYGISGYYIGVPCKLGARGLEQIIEIRLTPEEEAALKKSAEAVKELCAVIGV --seq_id query1 --output_dir /content/thermalgan_outputs --name query1 --replicates 5 --temperature 1.0 --hf_model facebook/esm1v_t33_650M_UR90S_1 --device cuda --batch_tokens 20000 --max_length 1022 --gpu 0 --esm_init_mode single --esm_random_mask_prob 0.15 --esm_random_chunk 8 --esm_random_seed 42 --filter_opt_cycles 0 --sampling_cycles 0 --esm_filter_threshold 3.0 --esm_bf16 --ogt_config /content/thermalgan_weights/config_classifier1.yaml --ogt_weights /content/thermalgan_weights/OGT/Model1/model_weights.h5 /content/thermalgan_weights/OGT/Model2/model_weights.h5 

In [8]:
#@title Inspect outputs and download
from pathlib import Path

fasta_path = OUTPUT_DIR / f"{output_name}__variants.fasta"
print("Output FASTA:", fasta_path)

if fasta_path.exists():
    !head -n 20 {fasta_path}
else:
    print("FASTA not found; check logs above.")

softmax_path = OUTPUT_DIR / f"{output_name}_variants_softmax.jsonl"
if softmax_path.exists():
    print("Softmax JSONL:", softmax_path)

try:
    from google.colab import files
    if fasta_path.exists():
        files.download(str(fasta_path))
    if softmax_path.exists():
        files.download(str(softmax_path))
except Exception as exc:
    print(f"Download skipped: {exc}")


Output FASTA: /content/thermalgan_outputs/query1__variants.fasta
>query1_wt_rep0 | type=wt | predicted_OGT=45.8273 | G_global_nll=0.1195 | G_sub_nll=NA | ESM_global_nll=0.7421 | ESM_sub_nll=NA | Identity_to_WT(%)=NA | temp=NA
MRKKVTIVGAGNVGATAAHWIASKELADVVLIDVVEGVPQGKGLDLLEAMPIEKRDAHILGTNDYADTANSDIVVITAGIARKPGMSRDDLLQTNYKIMSDVVGKAIAASPDCILIIVSNPLDAMAQTAYRQAKFNRNRVIGMAGVLDSARFRTFIAEELKVSVENVTAFVLGGHGDTMVPLARYSTVAGIPITELIPKERLDAIVQRTRDGGAEIVKYLKTGSAYYAPSAAAVEMVEAILKDKKKILPCAAFLEGEYGISGYYIGVPCKLGARGLEQIIEIRLTPEEEAALKKSAEAVKELCAVIGV
>query1_wt_rep1 | type=wt | predicted_OGT=45.8273 | G_global_nll=0.1195 | G_sub_nll=NA | ESM_global_nll=0.7421 | ESM_sub_nll=NA | Identity_to_WT(%)=NA | temp=NA
MRKKVTIVGAGNVGATAAHWIASKELADVVLIDVVEGVPQGKGLDLLEAMPIEKRDAHILGTNDYADTANSDIVVITAGIARKPGMSRDDLLQTNYKIMSDVVGKAIAASPDCILIIVSNPLDAMAQTAYRQAKFNRNRVIGMAGVLDSARFRTFIAEELKVSVENVTAFVLGGHGDTMVPLARYSTVAGIPITELIPKERLDAIVQRTRDGGAEIVKYLKTGSAYYAPSAAAVEMVEAILKDKKKILPCAAFLEGEYGISGYYIGVPCKLGARGLEQIIEIRLTPEEEAALKKSAEAVKELCA

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>